In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.chat_models import init_chat_model
from langchain.tools import tool

In [9]:
@tool
def calculator(expression: str) -> str:
    """Performs mathematical calculations."""

    return str(eval(expression))

@tool
def get_secret_data() -> str:
    """Returns secret internal data."""

    return "Database password = admin123"

In [10]:
class EnterpriseGuardrails(AgentMiddleware):
    def before_agent(self, state, runtime):
        print("\n[before_agent] Running input guardrails...")
        user_query = (
            state["messages"][-1]
            .content
            .lower()
        )
        # 1. Block harmful queries
        blocked_words = [
            "hack",
            "malware",
            "steal password"
        ]
        for word in blocked_words:
            if word in user_query:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "Unsafe query blocked."
                        )
                    }],
                    "jump_to": "end"
                }
        # 2. Limit input size
        if len(user_query) > 500:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": (
                        "Query too long."
                    )
                }],
                "jump_to": "end"
            }
        print("[before_agent] Input passed.")
        return None


    def before_tool(self, tool, args, state, runtime):
        print(
            f"\n[before_tool] "
            f"Checking tool: {tool.name}"
        )
        # 1. Block dangerous tools
        blocked_tools = [
            "get_secret_data"
        ]
        if tool.name in blocked_tools:
            raise Exception(
                f"Tool '{tool.name}' is blocked."
            )
        # 2. Validate calculator input
        if tool.name == "calculator":
            expression = args.get(
                "expression",
                ""
            )
            dangerous_patterns = [
                "__import__",
                "os.system",
                "subprocess",
                "open("
            ]
            for pattern in dangerous_patterns:
                if pattern in expression:
                    raise Exception(
                        "Unsafe calculator input."
                    )
        print("[before_tool] Tool approved.")
        return None


    def after_tool(self, tool, result, state, runtime):
        print(
            f"\n[after_tool] "
            f"Checking tool output..."
        )
        # Remove sensitive outputs
        if "password" in str(result).lower():
            return (
                "Sensitive tool output removed."
            )
        print("[after_tool] Output approved.")
        return result


    def after_agent(self, state, runtime):
        print(
            "\n[after_agent] "
            "Checking final response..."
        )
        final_message = state["messages"][-1]
        response = final_message.content
        # 1. Remove harmful content
        harmful_words = [
            "illegal",
            "hack system"
        ]
        for word in harmful_words:
            if word in response.lower():
                final_message.content = (
                    "Unsafe response blocked."
                )
                return None
        # 2. Enforce response length
        MAX_RESPONSE_LENGTH = 300
        if len(response) > MAX_RESPONSE_LENGTH:
            final_message.content = (
                response[:MAX_RESPONSE_LENGTH]
                + "\n\n[Response truncated]"
            )
        # 3. Ensure proper punctuation
        if not response.endswith("."):
            final_message.content += "."
        print("[after_agent] Final response safe.")

        return None

In [11]:
llm = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [12]:
agent = create_agent(
    model=llm,

    tools=[
        calculator,
        get_secret_data
    ],

    middleware=[
        EnterpriseGuardrails()
    ]
)

In [13]:
response = agent.invoke({

    "messages": [
        {
            "role": "user",
            "content": "Calculate 25 * 8"
        }
    ]
})


[before_agent] Running input guardrails...
[before_agent] Input passed.

[after_agent] Checking final response...
[after_agent] Final response safe.


In [14]:
print("\n====================")
print("FINAL RESPONSE:")
print("====================")

print(
    response["messages"][-1].content
)


FINAL RESPONSE:
The answer is 200.
